# Laikipia farm digitisation: train, compare and map on a Kaggle GPU

One run does everything:
1. trains one U-Net per experiment (E0, E1, E2), one after the other, each saved in its own folder;
2. scores every model on the held-out test blocks (object F1 against your hand-traced farms);
3. picks the best model and polygonises farms with it;
4. writes KMZ files (Google Earth Pro) and GeoPackages (QGIS) of its farms, for the test blocks and the whole
   working area, plus a comparison table of all experiments.

**Input to attach** (*Add Input → Your Work → Datasets*): one **private** dataset, `laikipia-s2`, holding

| File (from the laptop) | Used for |
|---|---|
| `tiles_s2_E0.zip`, `tiles_s2_E1.zip`, `tiles_s2_E2.zip` | training E0 / E1 / E2 |
| `S2_GEOMEDIAN_20250101_H1.tif` (from `data/imagery/scenes_s2/`) | scoring + KMZ |
| `dea.tif`, `gee.tif` (from `data/context/`) | scoring + KMZ for E1 / E2 |

Files are found by name anywhere inside the dataset, and zips Kaggle did not unpack are unpacked here.
Your parcels and working area come from the cloned GitHub repository.

**Settings:** *Accelerator → GPU T4 or P100*, *Internet → On*. Then **Save Version → Save & Run All (commit)**:
it runs in the background (up to 12 h) and keeps everything in `/kaggle/working` as the output.
Experiments already finished are skipped on a rerun; to continue after a session ended, attach the previous
version's output as an input (training resumes from each experiment's `last.pt`).

Without the scenes dataset, models are compared by validation IoU only and no KMZ is made.
Nothing needs to be edited between experiments: one **Save & Run All** does the whole sequence.

## 1. Settings

In [ ]:
from pathlib import Path

DATASET = "laikipia-s2"             # the Kaggle dataset holding the chips, the scene and the context layers

# experiment name -> chips folder / zip name inside the dataset (remove a line to skip that experiment)
EXPERIMENTS = {
    "E0": "tiles_s2_E0",            # Sentinel-2 image only
    "E1": "tiles_s2_E1",            # + Digital Earth Africa context
    "E2": "tiles_s2_E2",            # + Digital Earth Africa + Earth Engine context
}
CONFIG = "config_s2.yaml"           # config the chips were cut with (config.yaml for Legion, config_nicfi.yaml for NICFI)
SCENES = "S2_GEOMEDIAN_*.tif"       # the ingested scene(s), matched by file name
CONTEXT_FILES = ["dea.tif", "gee.tif"]
LABELS_DATASET = ""                 # optional: a dataset with other KMZ labels; default = the repo's data/labels/manual
MAP_AREA = "all"                    # "all" = KMZ of the whole working area too; "test" = test blocks only (faster)

REPO = "https://github.com/mangaorphy/Farm_Digitisation.git"
EPOCHS = 40
ENCODER = "resnet34"
BATCH = 16                          # lower to 8 if CUDA runs out of memory

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

## 2. Install and get the code

In [ ]:
!pip install -q "segmentation-models-pytorch>=0.3.4" rasterio geopandas pyogrio
import subprocess, sys
code_dir = WORK / "Farm_Digitisation"
if not code_dir.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(code_dir)], check=True)
PROJECT = code_dir / "laikipia-farm-mapper"
sys.path.insert(0, str(PROJECT))
import farmmap, torch
print("farmmap", farmmap.__version__, "| torch", torch.__version__, "| GPU:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on the GPU accelerator!")

## 3. Find the inputs

In [ ]:
import pandas as pd, zipfile, os
from farmmap import config

def dataset_dir(name):
    """Folder of an attached dataset (Kaggle mounts it under /kaggle/input/<name>, sometimes nested)."""
    hits = sorted((p for p in INPUT.rglob("*") if p.is_dir() and p.name == name), key=lambda p: len(p.parts))
    return hits[0] if hits else None

ROOT = dataset_dir(DATASET)
assert ROOT, f"dataset {DATASET!r} is not attached (Add Input); inputs: {[p.name for p in INPUT.iterdir()]}"
print("dataset:", ROOT)

def chips_dir(name):
    """Folder with index.csv below a folder called `name`; unzips `name`.zip first if Kaggle left it zipped."""
    def search(base):
        return sorted((p.parent for p in base.rglob("index.csv") if name in p.parent.parts),
                      key=lambda p: len(p.parts))
    found = search(ROOT)
    if not found:
        zips = sorted(ROOT.rglob(f"{name}.zip"))
        if zips:
            dest = WORK / "unzipped" / name
            if not dest.exists():
                print(f"unzipping {zips[0].name} ...")
                zipfile.ZipFile(zips[0]).extractall(dest)
            found = search(dest)
    return found[0] if found else None

tiles = {}
for exp, name in EXPERIMENTS.items():
    tiles[exp] = chips_dir(name)
    assert tiles[exp], f"{exp}: no chips folder or zip named {name!r} in {ROOT}"
    idx = pd.read_csv(tiles[exp] / "index.csv")
    bands = (tiles[exp] / "meta.yaml").read_text().split("bands:")[1].split("\n")[0].strip()
    print(f"{exp}: {len(idx)} chips {dict(idx.split.value_counts())}  bands {bands}\n    {tiles[exp]}")

scenes = sorted(ROOT.rglob(SCENES))
# context files may sit in different folders: link them into one folder for `paths.context`
CONTEXT_DIR = WORK / "context"; CONTEXT_DIR.mkdir(exist_ok=True)
context_tifs = []
for f in CONTEXT_FILES:
    hit = sorted(ROOT.rglob(f))
    if hit:
        link = CONTEXT_DIR / f
        if not link.exists():
            os.symlink(hit[0], link)
        context_tifs.append(link)
labels_dir = dataset_dir(LABELS_DATASET) if LABELS_DATASET else None
SCORE = bool(scenes)
print("scenes:", [str(p) for p in scenes] or "none -> compare by validation IoU, no KMZ")
print("context:", [p.name for p in context_tifs] or "none (only E0 can be scored)")

def make_cfg(exp):
    over = [f"paths.tiles={tiles[exp]}", f"paths.models={WORK / 'models' / exp}",
            f"paths.outputs={WORK / 'outputs' / exp}", f"train.epochs={EPOCHS}", f"train.batch_size={BATCH}",
            f"model.encoder={ENCODER}", "train.num_workers=4"]
    cfg = config.load(PROJECT / CONFIG, over)
    if labels_dir:
        cfg.paths.labels = labels_dir
    if context_tifs:
        cfg.paths.context = CONTEXT_DIR
    return cfg

A few chips of the first experiment (green = interior, red = boundary, grey = ignored):

In [ ]:
import numpy as np, rasterio, matplotlib.pyplot as plt

def show_rgb(img):
    return np.moveaxis(img[:3], 0, -1)

def overlay(mask):
    rgba = np.zeros(mask.shape + (4,))
    rgba[mask == 1] = (0, 1, 0, .35); rgba[mask == 2] = (1, 0, 0, .8); rgba[mask == 255] = (.5, .5, .5, .6)
    return rgba

first = next(iter(tiles.values()))
sample = pd.read_csv(first / "index.csv").sample(6, random_state=0)
fig, axes = plt.subplots(2, 6, figsize=(19, 6.4))
for k, tid in enumerate(sample.id):
    with rasterio.open(first / "images" / f"{tid}.tif") as s: img = s.read()
    with rasterio.open(first / "masks" / f"{tid}.tif") as s: m = s.read(1)
    axes[0, k].imshow(show_rgb(img)); axes[0, k].set_title(tid, fontsize=7)
    axes[1, k].imshow(show_rgb(img)); axes[1, k].imshow(overlay(m))
for a in axes.ravel(): a.axis("off")
plt.tight_layout()

## 4. Train every experiment, one after the other

In [ ]:
import shutil, time
from farmmap.train import train

best_pt = {}
for exp in EXPERIMENTS:
    cfg = make_cfg(exp)
    out = Path(cfg.paths.models); out.mkdir(parents=True, exist_ok=True)
    # continue from a previous version's output if it was attached as an input
    prev = [p for p in INPUT.rglob(f"models/{exp}/last.pt")]
    if prev and not (out / "last.pt").exists():
        print(f"{exp}: resuming from {prev[0]}")
        for f in prev[0].parent.glob("*"):
            shutil.copy(f, out / f.name)
    t0 = time.time()
    print(f"\n========== {exp}: training on {tiles[exp]} ==========")
    best_pt[exp] = train(cfg)
    print(f"{exp}: done in {(time.time() - t0) / 60:.0f} min")

## 5. Learning curves

In [ ]:
hist = {exp: pd.read_csv(WORK / "models" / exp / "history.csv") for exp in EXPERIMENTS}
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for exp, h in hist.items():
    ax[0].plot(h.epoch, h.val_loss, label=exp); ax[1].plot(h.epoch, h.val_miou, label=exp)
ax[0].set_title("validation loss"); ax[1].set_title("validation mIoU (interior + boundary)"); ax[1].set_ylim(0, 1)
for a in ax: a.legend(); a.set_xlabel("epoch")
pd.DataFrame({exp: {"best val mIoU": h.val_miou.max(), "best epoch": int(h.val_miou.idxmax()),
                    "epochs run": len(h)} for exp, h in hist.items()}).T

## 6. Score every model on the test blocks

Each model predicts the 2 km test blocks it never saw, its farms are polygonised and matched one to one with
your hand-traced farms (IoU ≥ 0.5). This is the real comparison; validation IoU is per pixel.

In [ ]:
from farmmap import imagery, predict, polygonize, evaluate, export
from farmmap.cli import _bounds
import types

rows = {}
if SCORE:
    mosaic = imagery.write_vrt(scenes, WORK / "mosaic.vrt")
    test_bounds = _bounds(make_cfg(next(iter(EXPERIMENTS))), types.SimpleNamespace(split="test", bbox=None))
    for exp in EXPERIMENTS:
        cfg = make_cfg(exp); cfg.paths.mosaic = mosaic
        outd = Path(cfg.paths.outputs); outd.mkdir(parents=True, exist_ok=True)
        print(f"\n========== {exp}: test blocks ==========")
        try:
            prob = predict.predict(cfg, best_pt[exp], mosaic, outd / "probabilities_test.tif", test_bounds)
        except FileNotFoundError as e:  # model needs context layers that were not attached
            print(f"{exp}: skipped - {e}"); continue
        polygonize.polygonize(cfg, prob, outd / "farms_test.gpkg")
        r = evaluate.evaluate(cfg, outd / "farms_test.gpkg", split="test")
        rows[exp] = {k: r.get(k) for k in ("f1", "precision", "recall", "predictions", "parcels", "matched",
                                           "median_best_iou", "merged_predictions", "over_segmented_parcels")}
        for size, v in (r.get("recall_by_size") or {}).items():
            rows[exp][f"recall {size}"] = v.get("recall") if isinstance(v, dict) else v

comparison = pd.DataFrame({exp: {"best val mIoU": round(h.val_miou.max(), 3)} for exp, h in hist.items()}).T
if rows:
    comparison = comparison.join(pd.DataFrame(rows).T)
comparison.to_csv(WORK / "comparison.csv")
comparison

## 7. Pick the best model

In [ ]:
scored = comparison.dropna(subset=["f1"]) if "f1" in comparison else comparison.iloc[0:0]
if len(scored):
    # simplest model within 0.02 F1 of the best (E0 < E1 < E2 in the order of EXPERIMENTS)
    top = scored.f1.astype(float).max()
    BEST = next(exp for exp in EXPERIMENTS if exp in scored.index and float(scored.f1[exp]) >= top - 0.02)
    print(f"Best by test F1: {BEST} (F1 {float(scored.f1[BEST]):.3f}; top F1 {top:.3f})")
else:
    BEST = comparison["best val mIoU"].astype(float).idxmax()
    print(f"No test scores (scenes not attached): best by validation mIoU = {BEST}")
best_dir = WORK / "best"; best_dir.mkdir(exist_ok=True)
shutil.copy(best_pt[BEST], best_dir / f"best_{BEST}.pt")

## 8. Farms from the best model → KMZ

Always writes `best/farms_best_test.kmz` (the test blocks, where your traced farms let you judge it).
With `MAP_AREA = "all"` (default) it also maps the whole working area into `best/farms_best_all.kmz`
(roughly 10–30 min of GPU plus polygonising on the CPU).

In [ ]:
import geopandas as gpd

def write_kmz(farms, stem, label):
    farms.to_file(best_dir / f"{stem}.gpkg", driver="GPKG")
    export.to_kmz(farms, best_dir / f"{stem}.kmz", name=f"Auto-detected farms ({BEST}, {label})")
    n_low = int((farms.score < 0.6).sum()) if len(farms) else 0
    print(f"{label}: {len(farms)} farms, {farms.area_ha.sum() if len(farms) else 0:.0f} ha, "
          f"{n_low} with score < 0.6 (orange, review first) -> {best_dir / (stem + '.kmz')}")

if SCORE:
    cfg = make_cfg(BEST); cfg.paths.mosaic = mosaic
    farms = gpd.read_file(Path(cfg.paths.outputs) / "farms_test.gpkg")   # from step 6
    write_kmz(farms, "farms_best_test", "test blocks")
    if MAP_AREA == "all":
        prob = predict.predict(cfg, best_pt[BEST], mosaic, best_dir / "probabilities_all.tif", None)
        farms_all = polygonize.polygonize(cfg, prob, best_dir / "farms_best_all.gpkg")
        write_kmz(farms_all, "farms_best_all", "whole working area")
else:
    print("Attach the scenes dataset to make the KMZ.")

Quick look: predicted farms (yellow) and your traced farms (cyan) on the busiest test area:

In [ ]:
if SCORE:
    from farmmap.labels import load_labels
    from rasterio.windows import from_bounds
    ref = load_labels(cfg)
    ref = ref[ref.SOURCE.eq("Digitized")] if "SOURCE" in ref else ref
    # centre on the densest 2 km cell of predicted farms (or of traced farms if none were predicted)
    pts = farms if len(farms) else ref.cx[test_bounds[0]:test_bounds[2], test_bounds[1]:test_bounds[3]]
    if len(pts) == 0:
        print("No predicted or traced farms in the test blocks to show.")
    else:
        c = pts.geometry.centroid
        cell = (np.floor(c.x / 2000) * 2000).astype(int).astype(str) + "_" + (np.floor(c.y / 2000) * 2000).astype(int).astype(str)
        x0, y0 = map(float, cell.value_counts().index[0].split("_")); x, y = x0 + 1000, y0 + 1000
        b = (x - 1000, y - 1000, x + 1000, y + 1000)
        with rasterio.open(mosaic) as s:
            img = s.read([1, 2, 3], window=from_bounds(*b, s.transform))
        fig, ax = plt.subplots(figsize=(9, 9))
        ax.imshow(np.moveaxis(img, 0, -1), extent=(b[0], b[2], b[1], b[3]))
        ref.cx[b[0]:b[2], b[1]:b[3]].boundary.plot(ax=ax, color="cyan", linewidth=0.8)
        farms.cx[b[0]:b[2], b[1]:b[3]].boundary.plot(ax=ax, color="yellow", linewidth=1.2)
        ax.set_xlim(b[0], b[2]); ax.set_ylim(b[1], b[3]); ax.set_title(f"{BEST}: predicted (yellow) vs traced (cyan), 2 x 2 km")
        ax.axis("off")

## 9. Download

From the notebook output (right panel, or the *Output* tab of the saved version), folder `best/`:

- `farms_best_all.kmz`: farms of the whole working area, for Google Earth Pro (green = confident,
  orange = score < 0.6, review first); `farms_best_all.gpkg` is the same for QGIS;
- `farms_best_test.kmz` / `.gpkg`: the test blocks only, to compare with your traced farms;
- `best_<E>.pt`: the winning model;

and at the top level:

- `comparison.csv`: scores of every experiment;
- `models/E0/best.pt`, `models/E1/best.pt`, `models/E2/best.pt`: every trained model (on the laptop put them
  in `data/models_s2_E0/` etc.).